# TrackNetV5 — Pickleball Ball Tracking

Implementation based on the TrackNetV5 paper.
Architecture: **MDD Module** → **V2 U-Net Backbone** → **R-STR Head** (Residual-driven Spatio-Temporal Refinement).

**Pipeline:**
- Input: 3 consecutive RGB frames → 13-channel MDD tensor
- Output: 3 heatmaps (one per frame) with ball probability at each pixel
- Loss: Weighted Binary Cross-Entropy (WBCE)
- Post-process: argmax on heatmap → (x, y) ball coordinate

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', '-q', 'opencv-python-headless', 'tqdm'])
print('Done.')

In [ ]:
import os, random, math, xml.etree.ElementTree as ET
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import MultiStepLR
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Configuration

In [ ]:
CLIP_DIRS = [
    '/content/drive/MyDrive/In_Out_Pickleball/game_1/clip_1/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_1/clip_2/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_1/clip_4/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_1/clip_5/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_2/clip_1/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_2/clip_2/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_2/clip_3/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_2/clip_4/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_2/clip_5/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_3/clip_1/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_3/clip_2/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_3/clip_3/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_3/clip_4/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_3/clip_5/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_4/Clip_1/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_4/Clip_2/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_4/Clip_3/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_4/Clip_4/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_4/Clip_5/frames',
    '/content/drive/MyDrive/In_Out_Pickleball/game_5/clip_1/frames',
]

# Original resolution — no downscale
INPUT_H = 1080
INPUT_W = 1920
# r=40px at 1920x1080 as stated in TrackNetV5 paper (Loveall dataset)
HEATMAP_RADIUS = 40

# PATCH_SIZE must divide both dims. 16 fails (1080/16=67.5). 24 works: 1080/24=45, 1920/24=80.
PATCH_SIZE = 24
D_MODEL    = 256
N_HEADS    = 8
N_LAYERS   = 2
MASK_PROB  = 0.1

# BATCH_SIZE=1 required: enc1 is [B,64,1080,1920] ~1GB/sample, batch>1 OOMs A100 80GB.
BATCH_SIZE        = 1
LEARNING_RATE     = 1e-4
NUM_EPOCHS        = 30
LR_DECAY_EPOCHS   = [20, 25]
LR_DECAY_GAMMA    = 0.1
VAL_RATIO         = 0.3
SEED              = 42
PATIENCE          = 10
DETECTION_THRESHOLD = 0.5
TOLERANCE_DIST    = 4

SAVE_DIR = '/content/drive/MyDrive/In_Out_Pickleball/model'
os.makedirs(SAVE_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f'Resolution : {INPUT_W}x{INPUT_H} | Heatmap radius: {HEATMAP_RADIUS}px')
print(f'Patch size : {PATCH_SIZE} -> {INPUT_W//PATCH_SIZE}x{INPUT_H//PATCH_SIZE} = {(INPUT_W//PATCH_SIZE)*(INPUT_H//PATCH_SIZE)} patches')
print(f'Batch size : {BATCH_SIZE} | Config loaded.')


## 3. Data Preprocessing

Parse `annotations.xml` (CVAT image-level format) from each clip and build (t-1, t, t+1) frame triples.

In [ ]:
def parse_cvat_xml(xml_path):
    """
    Parse CVAT image-level XML.
    Returns dict: {frame_id: (x, y, visibility, orig_w, orig_h)}
    x, y are absolute pixel coordinates in the original image resolution.
    visibility=1 if ball is annotated, 0 otherwise.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    annotations = {}
    for image in root.findall('image'):
        fid    = int(image.get('id'))
        orig_w = int(image.get('width'))
        orig_h = int(image.get('height'))
        pts    = image.find('points')
        if pts is not None:
            x, y = map(float, pts.get('points').split(','))
            vis  = 1
        else:
            x, y = -1.0, -1.0
            vis  = 0
        annotations[fid] = (x, y, vis, orig_w, orig_h)
    return annotations


def build_clip_samples(clip_dir):
    """
    Build consecutive (t-1, t, t+1) triples for one clip.
    Returns list of (frame_paths_triple, coords_triple).
    """
    xml_path = os.path.join(clip_dir, 'annotations.xml')
    if not os.path.exists(xml_path):
        print(f'  [SKIP] No annotations.xml: {clip_dir}')
        return []

    annotations = parse_cvat_xml(xml_path)
    if not annotations:
        return []

    # Sorted frame filenames → e.g. ['000000.jpg', '000001.jpg', ...]
    frame_files = sorted(
        f for f in os.listdir(clip_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    )
    if len(frame_files) < 3:
        return []

    # frame_ids[i] corresponds to frame_files[i]
    frame_ids = [int(os.path.splitext(f)[0]) for f in frame_files]

    # Default dimensions from first annotated entry
    first = next(iter(annotations.values()))
    def_w, def_h = first[3], first[4]

    # Fill unannotated frames as invisible
    full = {}
    for fid in frame_ids:
        full[fid] = annotations.get(fid, (-1.0, -1.0, 0, def_w, def_h))

    samples = []
    for i in range(1, len(frame_ids) - 1):
        paths  = [os.path.join(clip_dir, frame_files[j]) for j in (i-1, i, i+1)]
        coords = [full[frame_ids[j]] for j in (i-1, i, i+1)]
        samples.append((paths, coords))
    return samples


def build_all_samples(clip_dirs):
    all_samples = []
    for d in clip_dirs:
        print(f'Processing: {d}')
        s = build_clip_samples(d)
        print(f'  → {len(s)} triples')
        all_samples.extend(s)
    print(f'\nTotal samples: {len(all_samples)}')
    return all_samples


all_samples = build_all_samples(CLIP_DIRS)

# Shuffle then split
indices = list(range(len(all_samples)))
random.shuffle(indices)
n_val   = int(len(all_samples) * VAL_RATIO)
val_samples   = [all_samples[i] for i in indices[:n_val]]
train_samples = [all_samples[i] for i in indices[n_val:]]
print(f'Train: {len(train_samples)} | Val: {len(val_samples)}')

## 4. Dataset & DataLoader

## 4. Dataset Verification

Visually confirm that each image and its annotation from `annotations.xml` are correctly aligned.
Displays **3 consecutive frames** (t-1, t, t+1) — the same triplet structure used during training.

- **Green circle** = ball region used as training label (`HEATMAP_RADIUS = 40px`)
- **Red dot** = exact annotated coordinate (x, y) from CVAT
- **Red X** = frame marked as no-ball (`visibility=0`)

Run this before training to catch labelling errors or mismatched file paths.

In [ ]:
def check_dataset(samples, num_triples=3, clip_dir_filter=None):
    """
    Show consecutive (t-1, t, t+1) frame triples with ball annotations overlaid.
    Green circle = training heatmap label footprint (radius = HEATMAP_RADIUS).
    Red X        = no ball annotation (visibility=0).
    """
    pool = samples
    if clip_dir_filter:
        pool = [s for s in samples if clip_dir_filter in s[0][1]]
        print(f'Filtered: {len(pool)} triples matching "{clip_dir_filter}"')
    if not pool:
        print('No samples found.'); return

    # Pick num_triples consecutive samples to preserve temporal order
    start  = random.randint(0, max(0, len(pool) - num_triples))
    chosen = pool[start : start + num_triples]

    fig, axes = plt.subplots(num_triples, 3, figsize=(22, num_triples * 4))
    if num_triples == 1:
        axes = [axes]
    col_labels = ['t-1  (prev)', 't  (target)', 't+1  (next)']

    for row, (frame_paths, coords) in enumerate(chosen):
        for col, (fp, (x, y, vis, ow, oh)) in enumerate(zip(frame_paths, coords)):
            ax = axes[row][col]
            img = cv2.imread(fp)
            if img is None:
                ax.text(0.5, 0.5, f'FILE MISSING\n{os.path.basename(fp)}',
                        ha='center', va='center', color='red', fontsize=9,
                        transform=ax.transAxes)
                ax.set_title(f'{col_labels[col]} — FILE MISSING', color='red', fontsize=8)
                ax.axis('off'); continue

            disp = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()

            if vis == 1 and x >= 0 and y >= 0:
                cx, cy = int(round(x)), int(round(y))
                cv2.circle(disp, (cx, cy), HEATMAP_RADIUS + 6, (255, 220, 0),  2)  # yellow margin ring
                cv2.circle(disp, (cx, cy), HEATMAP_RADIUS,     (0,   230, 0), -1)  # green = heatmap label
                cv2.circle(disp, (cx, cy), 5,                  (255,   0, 0), -1)  # red   = raw coordinate
                info = f'ball @ ({cx}, {cy})'
                tc   = 'green'
            else:
                h, w = disp.shape[:2]
                cv2.line(disp, (w//2-50, h//2-50), (w//2+50, h//2+50), (220, 0, 0), 5)
                cv2.line(disp, (w//2+50, h//2-50), (w//2-50, h//2+50), (220, 0, 0), 5)
                info = 'no ball (vis=0)'
                tc   = 'red'

            ax.imshow(disp)
            parts = fp.replace('\\', '/').split('/')
            short = '/'.join(parts[-3:])
            ax.set_title(f'{col_labels[col]}\n{short}\n{info}', color=tc, fontsize=8)
            for spine in ax.spines.values():
                spine.set_edgecolor(tc); spine.set_linewidth(2)
            ax.axis('off')

    plt.suptitle(
        f'Dataset Check  |  {num_triples} consecutive triples  |  '
        f'Green = heatmap label (r={HEATMAP_RADIUS}px at {INPUT_W}x{INPUT_H})  |  Red X = no ball',
        fontsize=12, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.show()
    print(f'Showed triples [{start}..{start+len(chosen)-1}] of {len(pool)} total')


# Show 3 consecutive triples from the full dataset
check_dataset(all_samples, num_triples=3)

# To check a specific clip only:
# check_dataset(all_samples, num_triples=3, clip_dir_filter='game_1/clip_1')


In [ ]:
def create_heatmap(x, y, vis, orig_w, orig_h):
    """
    Create binary heatmap with a filled circle at the scaled ball position.
    Returns float32 numpy array [INPUT_H, INPUT_W] in [0, 1].
    """
    hm = np.zeros((INPUT_H, INPUT_W), dtype=np.float32)
    if vis == 1 and x >= 0 and y >= 0:
        cx = int(round(x * INPUT_W / orig_w))
        cy = int(round(y * INPUT_H / orig_h))
        cx = max(HEATMAP_RADIUS, min(INPUT_W - HEATMAP_RADIUS - 1, cx))
        cy = max(HEATMAP_RADIUS, min(INPUT_H - HEATMAP_RADIUS - 1, cy))
        cv2.circle(hm, (cx, cy), HEATMAP_RADIUS, 1.0, -1)
    return hm


class TrackNetDataset(Dataset):
    """
    Each item:
      frames   — [3, 3, H, W]  float32 in [0,1]   (3 consecutive RGB frames)
      heatmaps — [3, H, W]     float32 in {0,1}   (GT heatmap for each frame)
    """
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frame_paths, coords = self.samples[idx]
        frames, heatmaps = [], []

        for fp, (x, y, v, ow, oh) in zip(frame_paths, coords):
            img = cv2.imread(fp)
            if img is None:
                img = np.zeros((INPUT_H, INPUT_W, 3), dtype=np.uint8)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (INPUT_W, INPUT_H))
            frames.append(img.astype(np.float32) / 255.0)
            heatmaps.append(create_heatmap(x, y, v, ow, oh))

        # [3, H, W, 3] → [3, 3, H, W]
        frames_t   = torch.from_numpy(np.stack(frames)).permute(0, 3, 1, 2)
        heatmaps_t = torch.from_numpy(np.stack(heatmaps))   # [3, H, W]
        return frames_t, heatmaps_t


train_loader = DataLoader(TrackNetDataset(train_samples),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(TrackNetDataset(val_samples),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## 5. Model Architecture

### 5a. MDD Module (Motion Direction Decoupling)
Converts 3 RGB frames into a 13-channel input tensor:
`Concat(I_{t-1}, A_{t-1,t}, I_t, A_{t,t+1}, I_{t+1})`
where each `A` is a 2-channel attention map from signed polarity fields P+ (brightening) and P- (darkening).

In [ ]:
class AttentionMapping(nn.Module):
    """Learnable sigmoid: f(x; α, β) = sigmoid(α·x + β) → [0, 1]"""
    def __init__(self):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(1))
        self.beta  = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return torch.sigmoid(self.alpha * x + self.beta)


class MDDModule(nn.Module):
    """
    Motion Direction Decoupling.

    Input:  [B, 3, 3, H, W]  — batch of 3 consecutive RGB frames
    Output: [B, 13, H, W]    — 3+2+3+2+3 channel fusion

    Polarity decomposition:
      Δ  = grayscale(I_t) - grayscale(I_{t-1})
      P+ = ReLU(Δ)   — pixels brightening (ball arriving)
      P- = ReLU(-Δ)  — pixels darkening  (ball departing)
      A  = Concat(f(P+), f(P-))  → 2 channels
    """
    def __init__(self):
        super().__init__()
        self.attn = AttentionMapping()

    @staticmethod
    def _gray(rgb):
        return 0.299*rgb[:,0:1] + 0.587*rgb[:,1:2] + 0.114*rgb[:,2:3]

    def _motion_attn(self, fa, fb):
        delta   = self._gray(fb) - self._gray(fa)     # [B,1,H,W]
        p_plus  = F.relu(delta)
        p_minus = F.relu(-delta)
        return torch.cat([self.attn(p_plus), self.attn(p_minus)], dim=1)  # [B,2,H,W]

    def forward(self, frames):
        I_prev, I_curr, I_next = frames[:,0], frames[:,1], frames[:,2]
        A_prev = self._motion_attn(I_prev, I_curr)
        A_next = self._motion_attn(I_curr, I_next)
        return torch.cat([I_prev, A_prev, I_curr, A_next, I_next], dim=1)  # [B,13,H,W]

### 5b. TrackNetV2 Backbone (U-Net Encoder-Decoder)

In [ ]:
def _conv_block(in_ch, out_ch, n=2):
    layers = []
    for i in range(n):
        layers += [
            nn.Conv2d(in_ch if i == 0 else out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
    return nn.Sequential(*layers)


class TrackNetV2Backbone(nn.Module):
    """
    U-Net style VGG encoder-decoder.
    Input:  [B, 13, H, W]
    Output: [B, 3,  H, W]  -- draft heatmap logits (before sigmoid)

    NOTE: Uses F.interpolate(size=skip.shape[2:]) instead of fixed scale_factor=2
    to handle input dimensions that are not divisible by 2^depth (e.g. H=1080).
    Without this, MaxPool2d floor-divides odd sizes (135->67) and the subsequent
    2x upsample (67->134) mismatches the skip connection (135), causing a
    'Sizes of tensors must match' RuntimeError on cat().
    """
    def __init__(self, in_ch=13, out_ch=3):
        super().__init__()
        self.pool = nn.MaxPool2d(2, 2)

        # Encoder
        self.enc1 = _conv_block(in_ch, 64,  n=2)
        self.enc2 = _conv_block(64,  128,   n=2)
        self.enc3 = _conv_block(128, 256,   n=3)
        self.enc4 = _conv_block(256, 512,   n=3)
        self.enc5 = _conv_block(512, 512,   n=3)

        # Decoder (skip connection channels added at cat)
        self.dec4 = _conv_block(512+512, 512, n=2)
        self.dec3 = _conv_block(512+256, 256, n=2)
        self.dec2 = _conv_block(256+128, 128, n=2)
        self.dec1 = _conv_block(128+64,   64, n=2)

        self.head = nn.Conv2d(64, out_ch, 1)

    @staticmethod
    def _up(x, target):
        """Upsample x to exactly match target's (H, W) — avoids off-by-one on odd sizes."""
        return F.interpolate(x, size=target.shape[2:], mode='bilinear', align_corners=True)

    def forward(self, x):
        e1 = self.enc1(x)                         # [B, 64,  H,    W   ]
        e2 = self.enc2(self.pool(e1))             # [B, 128, H/2,  W/2 ]
        e3 = self.enc3(self.pool(e2))             # [B, 256, H/4,  W/4 ]
        e4 = self.enc4(self.pool(e3))             # [B, 512, H/8,  W/8 ]
        e5 = self.enc5(self.pool(e4))             # [B, 512, H/16, W/16]

        d4 = self.dec4(torch.cat([self._up(e5, e4), e4], dim=1))
        d3 = self.dec3(torch.cat([self._up(d4, e3), e3], dim=1))
        d2 = self.dec2(torch.cat([self._up(d3, e2), e2], dim=1))
        d1 = self.dec1(torch.cat([self._up(d2, e1), e1], dim=1))
        return self.head(d1)                      # [B, 3, H, W]


### 5c. R-STR Head (Residual-driven Spatio-Temporal Refinement)

Factorized spatio-temporal attention (TimeSformer-style) over draft heatmap patches.
Outputs a residual correction added to the draft. Uses **Stochastic Context Masking** (ρ=0.1) during training.

In [ ]:
class FactorizedAttnBlock(nn.Module):
    """One block of factorized temporal + spatial attention."""
    def __init__(self, d, n_heads, ffn_ratio=4, dropout=0.1):
        super().__init__()
        self.norm_t  = nn.LayerNorm(d)
        self.attn_t  = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.norm_s  = nn.LayerNorm(d)
        self.attn_s  = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.norm_ff = nn.LayerNorm(d)
        self.ffn     = nn.Sequential(
            nn.Linear(d, d * ffn_ratio), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d * ffn_ratio, d), nn.Dropout(dropout),
        )

    def forward(self, x):
        # x: [B, T, N, d]
        B, T, N, d = x.shape

        # Temporal attention: each spatial location attends across T frames
        xt = x.permute(0, 2, 1, 3).reshape(B*N, T, d)
        a, _ = self.attn_t(self.norm_t(xt), self.norm_t(xt), self.norm_t(xt))
        xt = (xt + a).reshape(B, N, T, d).permute(0, 2, 1, 3)  # [B,T,N,d]

        # Spatial attention: each frame attends across N patches
        xs = xt.reshape(B*T, N, d)
        a, _ = self.attn_s(self.norm_s(xs), self.norm_s(xs), self.norm_s(xs))
        xs = (xs + a).reshape(B, T, N, d)

        # FFN
        xf = xs.reshape(B*T*N, d)
        return (xf + self.ffn(self.norm_ff(xf))).reshape(B, T, N, d)


class RSTRHead(nn.Module):
    """
    R-STR: Residual-driven Spatio-Temporal Refinement.

    Input:  draft  [B, T, H, W]
    Output: residual [B, T, H, W]
    """
    def __init__(self, H, W, T=3, patch_size=16, d=256,
                 n_heads=8, n_layers=2, mask_prob=0.1):
        super().__init__()
        assert H % patch_size == 0 and W % patch_size == 0
        self.P   = patch_size
        self.Hp  = H // patch_size
        self.Wp  = W // patch_size
        self.N   = self.Hp * self.Wp
        self.T   = T
        self.mask_prob = mask_prob
        P2 = patch_size * patch_size

        self.patch_embed = nn.Linear(P2, d)
        self.pos_embed   = nn.Parameter(torch.zeros(1, T, self.N, d))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.blocks = nn.ModuleList([FactorizedAttnBlock(d, n_heads) for _ in range(n_layers)])
        self.norm   = nn.LayerNorm(d)
        self.decode = nn.Linear(d, P2)

    def _patchify(self, x):
        B, T, H, W = x.shape
        P = self.P
        x = x.reshape(B, T, H//P, P, W//P, P)
        x = x.permute(0, 1, 2, 4, 3, 5)        # [B, T, Hp, Wp, P, P]
        return x.reshape(B, T, self.N, P*P)

    def _unpatchify(self, p, H, W):
        B, T, N, PP = p.shape
        P = self.P
        x = p.reshape(B, T, H//P, W//P, P, P)
        x = x.permute(0, 1, 2, 4, 3, 5)        # [B, T, Hp, P, Wp, P]
        return x.reshape(B, T, H, W)

    def forward(self, draft, is_training=False):
        B, T, H, W = draft.shape
        inp = draft

        # Stochastic Context Masking: randomly zero one context frame during training
        if is_training and self.mask_prob > 0 and random.random() < self.mask_prob:
            inp = draft.clone()
            inp[:, random.choice([0, 2])] = 0.0

        x = self.patch_embed(self._patchify(inp)) + self.pos_embed  # [B,T,N,d]
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x.reshape(B*T*self.N, -1)).reshape(B, T, self.N, -1)
        return self._unpatchify(self.decode(x), H, W)   # [B, T, H, W]

### 5d. Full TrackNetV5

In [ ]:
class TrackNetV5(nn.Module):
    """
    TrackNetV5 = MDD → V2 Backbone → R-STR

    Input:  frames  [B, 3, 3, H, W]   (3 consecutive RGB frames)
    Output: pred    [B, 3, H, W]      (ball probability heatmaps, sigmoid applied)
    """
    def __init__(self):
        super().__init__()
        self.mdd      = MDDModule()
        self.backbone = TrackNetV2Backbone(in_ch=13, out_ch=3)
        self.rstr     = RSTRHead(
            INPUT_H, INPUT_W, T=3,
            patch_size=PATCH_SIZE, d=D_MODEL,
            n_heads=N_HEADS, n_layers=N_LAYERS,
            mask_prob=MASK_PROB
        )

    def forward(self, frames, is_training=False):
        x        = self.mdd(frames)                       # [B, 13, H, W]
        draft    = self.backbone(x)                       # [B, 3,  H, W]  logits
        residual = self.rstr(draft, is_training)          # [B, 3,  H, W]
        return torch.sigmoid(draft + residual)            # [B, 3,  H, W]  ∈ [0,1]


model = TrackNetV5().to(device)

n_total = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_total:,}  (paper reports ~14.77M)')

# Shape verification
with torch.no_grad():
    dummy = torch.zeros(1, 3, 3, INPUT_H, INPUT_W, device=device)
    out   = model(dummy)
    print(f'Output shape: {tuple(out.shape)}  (expected: (1, 3, {INPUT_H}, {INPUT_W}))')

## 6. Loss, Metrics & Utilities

In [ ]:
def weighted_bce_loss(pred, target):
    """
    Weighted Binary Cross-Entropy (WBCE) from TrackNetV5 paper:
      L = -(1/N) Σ [(1-p)² · y·log(p) + p² · (1-y)·log(1-p)]

    Adaptive weights focus on hard examples and handle the extreme
    foreground/background imbalance (ball pixels ≪ background).
    """
    eps  = 1e-7
    pred = pred.clamp(eps, 1.0 - eps)
    loss = -((1 - pred).pow(2) * target * torch.log(pred) +
              pred.pow(2) * (1 - target) * torch.log(1 - pred))
    return loss.mean()


def heatmap_to_coord(hm, threshold=DETECTION_THRESHOLD):
    """Find peak location in heatmap. Returns (cx, cy) or None."""
    if hm.max() < threshold:
        return None
    r, c = np.unravel_index(np.argmax(hm), hm.shape)
    return (c, r)   # (x, y)


def compute_tp_fp_fn(pred, gt, thr=DETECTION_THRESHOLD, tol=TOLERANCE_DIST):
    """Compute TP/FP/FN for a batch. TP = detected within `tol` pixels of GT."""
    pred_np = pred.detach().cpu().numpy()
    gt_np   = gt.detach().cpu().numpy()
    tp = fp = fn = 0
    for b in range(pred_np.shape[0]):
        for t in range(pred_np.shape[1]):
            pc = heatmap_to_coord(pred_np[b, t], thr)
            gc = heatmap_to_coord(gt_np[b, t],   threshold=0.5)
            if gc and pc:
                if math.hypot(pc[0]-gc[0], pc[1]-gc[1]) <= tol:
                    tp += 1
                else:
                    fp += 1; fn += 1
            elif gc and not pc:
                fn += 1
            elif not gc and pc:
                fp += 1
    return tp, fp, fn


def f1_from_counts(tp, fp, fn):
    p  = tp / (tp + fp + 1e-9)
    r  = tp / (tp + fn + 1e-9)
    f1 = 2*p*r / (p + r + 1e-9)
    return {'precision': p, 'recall': r, 'f1': f1}


class EarlyStopping:
    def __init__(self, patience=PATIENCE, delta=1e-4):
        self.patience  = patience
        self.delta     = delta
        self.best      = None
        self.counter   = 0
        self.triggered = False

    def step(self, val_f1):
        if self.best is None or val_f1 > self.best + self.delta:
            self.best    = val_f1
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.triggered = True
        return self.triggered


print('Loss, metrics and utilities defined.')

## 7. Training & Validation Loop

In [ ]:
def train_epoch(model, loader, optimizer, epoch):
    model.train()
    total_loss = 0.0
    tp = fp = fn = 0
    for frames, heatmaps in tqdm(loader, desc=f'Train {epoch}'):
        frames   = frames.to(device)
        heatmaps = heatmaps.to(device)
        optimizer.zero_grad()
        pred = model(frames, is_training=True)
        loss = weighted_bce_loss(pred, heatmaps)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        b_tp, b_fp, b_fn = compute_tp_fp_fn(pred, heatmaps)
        tp += b_tp; fp += b_fp; fn += b_fn
    return total_loss / len(loader), f1_from_counts(tp, fp, fn)


@torch.no_grad()
def val_epoch(model, loader, epoch):
    model.eval()
    total_loss = 0.0
    tp = fp = fn = 0
    for frames, heatmaps in tqdm(loader, desc=f'Val   {epoch}'):
        frames   = frames.to(device)
        heatmaps = heatmaps.to(device)
        pred     = model(frames, is_training=False)
        total_loss += weighted_bce_loss(pred, heatmaps).item()
        b_tp, b_fp, b_fn = compute_tp_fp_fn(pred, heatmaps)
        tp += b_tp; fp += b_fp; fn += b_fn
    return total_loss / len(loader), f1_from_counts(tp, fp, fn)


print('Train/val functions defined.')

## 8. Main Training

In [ ]:
optimizer      = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler      = MultiStepLR(optimizer, milestones=LR_DECAY_EPOCHS, gamma=LR_DECAY_GAMMA)
early_stopping = EarlyStopping(patience=PATIENCE)

history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
best_f1 = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_m = train_epoch(model, train_loader, optimizer, epoch)
    va_loss, va_m = val_epoch(model, val_loader, epoch)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_f1'].append(tr_m['f1'])
    history['val_f1'].append(va_m['f1'])

    print(f'\nEpoch {epoch:02d}/{NUM_EPOCHS}  LR={scheduler.get_last_lr()[0]:.2e}')
    print(f'  Train  loss={tr_loss:.4f}  P={tr_m["precision"]:.4f}  '
          f'R={tr_m["recall"]:.4f}  F1={tr_m["f1"]:.4f}')
    print(f'  Val    loss={va_loss:.4f}  P={va_m["precision"]:.4f}  '
          f'R={va_m["recall"]:.4f}  F1={va_m["f1"]:.4f}')

    # Save best model
    if va_m['f1'] > best_f1:
        best_f1 = va_m['f1']
        torch.save({
            'epoch':      epoch,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_f1':     best_f1,
            'config': {
                'INPUT_H': INPUT_H, 'INPUT_W': INPUT_W,
                'PATCH_SIZE': PATCH_SIZE, 'D_MODEL': D_MODEL,
                'N_HEADS': N_HEADS, 'N_LAYERS': N_LAYERS,
            }
        }, os.path.join(SAVE_DIR, 'best.pt'))
        print(f'  ✓ Saved best model (F1={best_f1:.4f})')

    # Checkpoint every 5 epochs
    if epoch % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_f1': va_m['f1'],
        }, os.path.join(SAVE_DIR, f'epoch_{epoch:02d}.pt'))
        print(f'  Checkpoint saved: epoch_{epoch:02d}.pt')

    if early_stopping.step(va_m['f1']):
        print(f'\n⚠ Early stopping at epoch {epoch} '
              f'(no F1 improvement for {PATIENCE} epochs)')
        break

print(f'\nTraining complete. Best Val F1: {best_f1:.4f}')

## 9. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True)

ax2.plot(history['train_f1'], label='Train')
ax2.plot(history['val_f1'],   label='Val')
ax2.set_title('F1 Score'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True)

plt.tight_layout()
curve_path = os.path.join(SAVE_DIR, 'training_curves.png')
plt.savefig(curve_path, dpi=120)
plt.show()
print(f'Saved: {curve_path}')

## 10. Inference on Video

Load best checkpoint and run tracking on a test video.  
Uses a sliding window of 3 frames; detects ball in the middle frame.

In [ ]:
def load_model(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    m = TrackNetV5().to(device)
    m.load_state_dict(ckpt['model_state'])
    m.eval()
    print(f'Loaded epoch {ckpt["epoch"]}  (Val F1={ckpt["val_f1"]:.4f})')
    return m


def infer_video(model, video_path, output_path, trail_len=30):
    """
    Run TrackNetV5 on a video. Draws ball position and motion trail.
    Args:
        model:       loaded TrackNetV5
        video_path:  input .mp4
        output_path: output .mp4
        trail_len:   number of past positions shown as fading trail
    """
    cap = cv2.VideoCapture(video_path)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out = cv2.VideoWriter(output_path,
                          cv2.VideoWriter_fourcc(*'mp4v'),
                          fps, (orig_w, orig_h))

    def preprocess(bgr):
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        sml = cv2.resize(rgb, (INPUT_W, INPUT_H))
        t   = torch.from_numpy(sml.astype(np.float32) / 255.0).permute(2, 0, 1)
        return rgb, t  # original RGB, tensor [3,H,W]

    buf = []   # (orig_rgb, tensor)
    trail = []  # past (cx, cy) in original resolution

    with tqdm(total=total, desc='Inference') as pbar:
        while True:
            ret, bgr = cap.read()
            if not ret:
                break
            buf.append(preprocess(bgr))

            if len(buf) < 3:
                pbar.update(1)
                continue

            frames_t = torch.stack([b[1] for b in buf[-3:]])  # [3,3,H,W]
            frames_t = frames_t.unsqueeze(0).to(device)        # [1,3,3,H,W]

            with torch.no_grad():
                pred = model(frames_t)   # [1,3,H,W]

            hm    = pred[0, 1].cpu().numpy()   # middle-frame heatmap
            coord = heatmap_to_coord(hm)

            display = buf[-2][0].copy()  # middle frame original RGB

            # Draw fading trail
            for k, (tx, ty) in enumerate(trail[-trail_len:]):
                alpha = int(255 * (k + 1) / min(len(trail), trail_len))
                cv2.circle(display, (tx, ty), 4, (255, 165, 0), -1)

            if coord is not None:
                cx = int(coord[0] * orig_w / INPUT_W)
                cy = int(coord[1] * orig_h / INPUT_H)
                conf = float(hm[coord[1], coord[0]])
                cv2.circle(display, (cx, cy), 10, (0, 255, 0), 2)
                cv2.circle(display, (cx, cy),  3, (255, 0,   0), -1)
                cv2.putText(display, f'ball {conf:.2f}', (cx+12, cy-8),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)
                trail.append((cx, cy))

            out.write(cv2.cvtColor(display, cv2.COLOR_RGB2BGR))
            pbar.update(1)

    cap.release(); out.release()
    print(f'Output: {output_path}')


# ── Usage ─────────────────────────────────────────────────────────────────────
# best_model = load_model(os.path.join(SAVE_DIR, 'best.pt'))
# infer_video(
#     best_model,
#     video_path='/content/drive/MyDrive/In_Out_Pickleball/vid_test_model/test.mp4',
#     output_path='/content/drive/MyDrive/In_Out_Pickleball/output/tracked.mp4',
# )